# UniAdvisor SLM training (Kaggle)
LoRA fine-tuning of `cross-encoder/mmarco-mMiniLMv2-L12-H384-v1` with typed heads (bool / choice / ordinal), temperature calibration and evaluation.

Settings: **GPU on, Internet on**. Input: the `uniadvisor-bundle` dataset (see kaggle/README.md).

In [ ]:
import glob, os, shutil, zipfile
# Kaggle usually unzips an uploaded archive into a folder; fall back to extracting a real .zip file
dirs = [os.path.dirname(p) for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
        if os.path.isfile(os.path.join(os.path.dirname(p), 'data', 'slm', 'train.jsonl'))]
zips = [p for p in glob.glob('/kaggle/input/**/*.zip', recursive=True) if os.path.isfile(p) and zipfile.is_zipfile(p)]
if dirs:
    shutil.copytree(dirs[0], '/kaggle/working/uniadvisor', dirs_exist_ok=True)
elif zips:
    zipfile.ZipFile(zips[0]).extractall('/kaggle/working')
else:
    raise FileNotFoundError('uniadvisor bundle not found under /kaggle/input: add the dataset as notebook input')
os.chdir('/kaggle/working/uniadvisor')
print(os.getcwd(), os.listdir('data/slm'))

In [ ]:
!pip install -q -e . --no-deps
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# one process per GPU (Kaggle 'GPU T4 x2' -> 2). --bs is per GPU: the effective batch is bs x GPUs.
# Watch the 'VRAM peak' line after ~20 steps: raise --bs (and --lr a little) while it stays under ~85%,
# lower --bs if you get CUDA out of memory.
import torch
n_gpu = max(1, torch.cuda.device_count())
print('GPUs:', n_gpu)
!torchrun --standalone --nproc_per_node={n_gpu} -m uniadvisor.slm.train --data data/slm --out /kaggle/working/slm --epochs 5 --bs 128 --lr 1e-3 --lora-r 16

In [ ]:
import json
m = json.load(open('/kaggle/working/slm/metrics.json'))
print(json.dumps(m['test'], indent=1, ensure_ascii=False))
print(json.dumps(m['temperatures'], indent=1), json.dumps(m['thresholds'], indent=1))

In [ ]:
shutil.make_archive('/kaggle/working/slm_model', 'zip', '/kaggle/working/slm')
print('download /kaggle/working/slm_model.zip and unzip into models/slm/')